# **1. Perkenalan Dataset**

Dataset yang digunakan adalah **Teen Social Media Usage & Mental Health** yang diperoleh dari Kaggle.

**Deskripsi Dataset:**
- **Sumber**: Kaggle (public repository)
- **Jumlah baris**: 2.500
- **Jumlah kolom**: 12
- **Tujuan**: Memprediksi risiko depresi (`depression_risk`: low/medium/high) pada remaja berdasarkan kebiasaan penggunaan media sosial dan faktor gaya hidup.

**Kolom Dataset:**
| Kolom | Tipe | Deskripsi |
|---|---|---|
| age | Integer | Usia remaja |
| gender | String | Jenis kelamin (male/female) |
| daily_social_media_hours | Float | Jam penggunaan medsos per hari |
| platform_usage | String | Platform yang digunakan (Instagram/TikTok/Both/Other) |
| sleep_hours | Float | Jam tidur per hari |
| screen_time_before_sleep | Float | Jam layar sebelum tidur |
| academic_performance | Float | Performa akademik |
| physical_activity | Float | Aktivitas fisik (jam/hari) |
| social_interaction_level | String | Level interaksi sosial (low/medium/high) |
| stress_level | Integer | Level stres (1-10) |
| anxiety_level | Integer | Level kecemasan (1-10) |
| depression_risk | String | **Target** - Risiko depresi (low/medium/high) |

# **2. Import Library**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.model_selection import train_test_split

import warnings
warnings.filterwarnings('ignore')

print('Libraries imported successfully!')

# **3. Memuat Dataset**

In [ ]:
df = pd.read_csv('Teen_Mental_Health_Dataset.csv')

print('Shape dataset:', df.shape)
print('\n--- 5 Baris Pertama ---')
df.head()

In [ ]:
print('--- Informasi Dataset ---')
df.info()

In [ ]:
print('--- Statistik Deskriptif ---')
df.describe()

# **4. Exploratory Data Analysis (EDA)**

In [ ]:
# 4.1 Cek missing values
print('=== Missing Values ===')
print(df.isnull().sum())
print('\nTotal missing values:', df.isnull().sum().sum())

In [ ]:
# 4.2 Cek duplikasi
print('=== Data Duplikat ===')
print('Jumlah duplikat:', df.duplicated().sum())

In [ ]:
# 4.3 Distribusi target variable
plt.figure(figsize=(8, 5))
order = ['low', 'medium', 'high']
colors = ['#2ecc71', '#f39c12', '#e74c3c']
df['depression_risk'].value_counts()[order].plot(kind='bar', color=colors, edgecolor='black')
plt.title('Distribusi Depression Risk', fontsize=14)
plt.xlabel('Depression Risk Level')
plt.ylabel('Jumlah')
plt.xticks(rotation=0)
for i, v in enumerate(df['depression_risk'].value_counts()[order]):
    plt.text(i, v + 10, str(v), ha='center', fontweight='bold')
plt.tight_layout()
plt.show()
print(df['depression_risk'].value_counts())

In [ ]:
# 4.4 Distribusi fitur numerik
num_cols = ['age', 'daily_social_media_hours', 'sleep_hours', 'screen_time_before_sleep',
            'academic_performance', 'physical_activity', 'stress_level', 'anxiety_level']

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()
for i, col in enumerate(num_cols):
    axes[i].hist(df[col], bins=20, color='steelblue', edgecolor='black', alpha=0.7)
    axes[i].set_title(col)
    axes[i].set_xlabel('Nilai')
    axes[i].set_ylabel('Frekuensi')
plt.suptitle('Distribusi Fitur Numerik', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# 4.5 Distribusi fitur kategorikal
cat_cols = ['gender', 'platform_usage', 'social_interaction_level']
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for i, col in enumerate(cat_cols):
    df[col].value_counts().plot(kind='bar', ax=axes[i], color='steelblue', edgecolor='black')
    axes[i].set_title(f'Distribusi {col}')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Jumlah')
    axes[i].tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# 4.6 Korelasi antar fitur numerik
plt.figure(figsize=(10, 8))
corr = df[num_cols].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=0.5)
plt.title('Heatmap Korelasi Fitur Numerik', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# 4.7 Boxplot: stress_level dan anxiety_level vs depression_risk
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
order = ['low', 'medium', 'high']
sns.boxplot(data=df, x='depression_risk', y='stress_level', order=order, ax=axes[0], palette='RdYlGn_r')
axes[0].set_title('Stress Level vs Depression Risk')
sns.boxplot(data=df, x='depression_risk', y='anxiety_level', order=order, ax=axes[1], palette='RdYlGn_r')
axes[1].set_title('Anxiety Level vs Depression Risk')
plt.tight_layout()
plt.show()

In [ ]:
# 4.8 Social media hours vs depression_risk
plt.figure(figsize=(8, 5))
order = ['low', 'medium', 'high']
sns.boxplot(data=df, x='depression_risk', y='daily_social_media_hours', order=order, palette='Blues')
plt.title('Jam Medsos Harian vs Depression Risk')
plt.tight_layout()
plt.show()

# **5. Data Preprocessing**

In [ ]:
# 5.1 Salin dataframe agar data asli tidak berubah
df_clean = df.copy()
print('Shape awal:', df_clean.shape)

In [ ]:
# 5.2 Tangani Missing Values
print('Missing values sebelum:', df_clean.isnull().sum().sum())

# Isi missing nilai numerik dengan median
for col in num_cols:
    if df_clean[col].isnull().sum() > 0:
        df_clean[col].fillna(df_clean[col].median(), inplace=True)

# Isi missing nilai kategorikal dengan modus
for col in cat_cols:
    if df_clean[col].isnull().sum() > 0:
        df_clean[col].fillna(df_clean[col].mode()[0], inplace=True)

print('Missing values sesudah:', df_clean.isnull().sum().sum())

In [ ]:
# 5.3 Hapus Duplikasi
print('Duplikat sebelum:', df_clean.duplicated().sum())
df_clean.drop_duplicates(inplace=True)
df_clean.reset_index(drop=True, inplace=True)
print('Duplikat sesudah:', df_clean.duplicated().sum())
print('Shape setelah hapus duplikat:', df_clean.shape)

In [ ]:
# 5.4 Encoding Fitur Kategorikal
le = LabelEncoder()

# Encode kolom-kolom kategorikal
df_clean['gender_encoded'] = le.fit_transform(df_clean['gender'])
df_clean['platform_encoded'] = le.fit_transform(df_clean['platform_usage'])
df_clean['social_interaction_encoded'] = le.fit_transform(df_clean['social_interaction_level'])

# Encode target variable
label_map = {'low': 0, 'medium': 1, 'high': 2}
df_clean['depression_risk_encoded'] = df_clean['depression_risk'].map(label_map)

print('Encoding selesai!')
print(df_clean[['gender', 'gender_encoded', 'platform_usage', 'platform_encoded']].head())

In [ ]:
# 5.5 Normalisasi Fitur Numerik
scaler = MinMaxScaler()
cols_to_scale = ['age', 'daily_social_media_hours', 'sleep_hours',
                 'screen_time_before_sleep', 'academic_performance',
                 'physical_activity', 'stress_level', 'anxiety_level']

df_clean[cols_to_scale] = scaler.fit_transform(df_clean[cols_to_scale])
print('Normalisasi selesai!')
df_clean[cols_to_scale].describe()

In [ ]:
# 5.6 Pilih kolom untuk dataset final
feature_cols = cols_to_scale + ['gender_encoded', 'platform_encoded', 'social_interaction_encoded']
target_col = 'depression_risk_encoded'

df_final = df_clean[feature_cols + [target_col]].copy()
print('Shape dataset final:', df_final.shape)
df_final.head()

In [ ]:
# 5.7 Simpan dataset yang sudah dipreprocessing
df_final.to_csv('Teen_Mental_Health_preprocessed.csv', index=False)
print('Dataset preprocessed berhasil disimpan!')
print('File: Teen_Mental_Health_preprocessed.csv')
print('Shape:', df_final.shape)